# Notebook 02: Data Cleaning & Feature Engineering

## Purpose

Transform raw DataCo dataset into a clean, analysis-ready format based on findings from Notebook 01.
- Every single row represents details related to one particular item/product in an order. 
## Cleaning Plan

1. **Drop columns** — redundant, null, or non-analytical
2. **Rename columns** — convert to snake_case
3. **Fix data types** — convert date columns
4. **Handle nulls** — contextual decisions per column
5. **Feature engineering** — create derived columns for analysis
6. **Validate** — sanity checks before saving
7. **Save** — output to `data/processed/dataco_cleaned.csv`

In [1]:
import pandas as pd
import numpy as np

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:,.2f}'.format)

# Load raw data
df = pd.read_csv(
    '../data/raw/DataCoSupplyChainDataset.csv',
    encoding='ISO-8859-1'
)

print(f"✅ Raw data loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

✅ Raw data loaded: 180,519 rows × 53 columns


### 1. Dropping Redundant Columns

 Following GDPR/CCPA principles by default — dropping PII when not needed, using surrogate IDs for tracking, and aggregating data before sharing with stakeholders

In [2]:
cols_to_drop = [
    # Duplicate columns (verified with .equals() in Notebook 01)
    'Order Customer Id',        # duplicate of Customer Id
    'Order Item Total',         # duplicate of Sales per customer
    'Order Profit Per Order',   # duplicate of Benefit per order
    'Order Item Cardprod Id',   # duplicate of Product Card Id
    'Product Category Id',      # duplicate of Category Id
    'Order Item Product Price', # duplicate of Product Price
    
    # No analytical value
    'Product Description',      # 100% null
    'Product Image',            # URL, not useful
    #Remove all presonally identifiable information (PII) columns
    'Customer Password',        # anonymized
    'Customer Email',           # anonymized
    'Customer Street',          # anonymized
    'Customer Fname',           # anonymized (XXXXX)
    'Customer Lname',           # anonymized
    'Latitude',                 
    'Longitude']              

df_clean = df.drop(columns=cols_to_drop)

print(f"Dropped {len(cols_to_drop)} columns")
print(f"Shape: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")

Dropped 15 columns
Shape: 180,519 rows × 38 columns


### 2. Renaming to Snake Case

In [3]:
# Rename to snake_case
rename_map = {
    'Type': 'payment_type',
    'Days for shipping (real)': 'days_shipping_real',
    'Days for shipment (scheduled)': 'days_shipping_scheduled',
    'Benefit per order': 'profit_per_order',
    'Sales per customer': 'sales_per_customer',
    'Delivery Status': 'delivery_status',
    'Late_delivery_risk': 'late_delivery_risk',
    'Category Id': 'category_id',
    'Category Name': 'category_name',
    'Customer City': 'customer_city',
    'Customer Country': 'customer_country',
    'Customer Id': 'customer_id',
    'Customer Segment': 'customer_segment',
    'Customer State': 'customer_state',
    'Customer Zipcode': 'customer_zipcode',
    'Department Id': 'department_id',
    'Department Name': 'department_name',
    'Latitude': 'latitude',
    'Longitude': 'longitude',
    'Market': 'market',
    'Order City': 'order_city',
    'Order Country': 'order_country',
    'order date (DateOrders)': 'order_date',
    'Order Id': 'order_id',
    'Order Item Discount': 'order_item_discount',
    'Order Item Discount Rate': 'order_item_discount_rate',
    'Order Item Id': 'order_item_id',
    'Order Item Profit Ratio': 'order_item_profit_ratio',
    'Order Item Quantity': 'order_item_quantity',
    'Sales': 'sales',
    'Order Region': 'order_region',
    'Order State': 'order_state',
    'Order Status': 'order_status',
    'Order Zipcode': 'order_zipcode',
    'Product Card Id': 'product_card_id',
    'Product Name': 'product_name',
    'Product Price': 'product_price',
    'Product Status': 'product_status',
    'shipping date (DateOrders)': 'shipping_date',
    'Shipping Mode': 'shipping_mode',
}

df_clean = df_clean.rename(columns=rename_map)

print(f"Renamed {len(rename_map)} columns ✅")
print(f"New columns sample: {list(df_clean.columns)}")

Renamed 40 columns ✅
New columns sample: ['payment_type', 'days_shipping_real', 'days_shipping_scheduled', 'profit_per_order', 'sales_per_customer', 'delivery_status', 'late_delivery_risk', 'category_id', 'category_name', 'customer_city', 'customer_country', 'customer_id', 'customer_segment', 'customer_state', 'customer_zipcode', 'department_id', 'department_name', 'market', 'order_city', 'order_country', 'order_date', 'order_id', 'order_item_discount', 'order_item_discount_rate', 'order_item_id', 'order_item_profit_ratio', 'order_item_quantity', 'sales', 'order_region', 'order_state', 'order_status', 'order_zipcode', 'product_card_id', 'product_name', 'product_price', 'product_status', 'shipping_date', 'shipping_mode']


#### Understanding relaitonship between all the money columns

In [4]:
money_cols = [
    'sales',
    'profit_per_order',
    'sales_per_customer',
    'order_item_discount',
    'order_item_discount_rate',
    'order_item_profit_ratio',
    'product_price',
    'order_item_quantity',
]

money_df = df_clean[money_cols]



In [5]:
# Check the relationship between sales, product_price, and order_item_quantity
relationship_check = df_clean[['product_price', 'order_item_quantity', 'sales']].copy()
relationship_check['expected_sales'] = (
    relationship_check['product_price'] * relationship_check['order_item_quantity']
)
relationship_check['matches'] = np.isclose(
    relationship_check['sales'], 
    relationship_check['expected_sales'],
    atol=0.01  # tolerance = 1 cent
)

print("Relationship Analysis:")
print(f"✅ All rows match (sales = product_price × quantity): {relationship_check['matches'].all()}")
print(f"\nMatches: {relationship_check['matches'].sum():,} / {len(relationship_check):,}")

print("\nSample rows:")
print(relationship_check.head(10))



Relationship Analysis:
✅ All rows match (sales = product_price × quantity): True

Matches: 180,519 / 180,519

Sample rows:
   product_price  order_item_quantity  sales  expected_sales  matches
0         327.75                    1 327.75          327.75     True
1         327.75                    1 327.75          327.75     True
2         327.75                    1 327.75          327.75     True
3         327.75                    1 327.75          327.75     True
4         327.75                    1 327.75          327.75     True
5         327.75                    1 327.75          327.75     True
6         327.75                    1 327.75          327.75     True
7         327.75                    1 327.75          327.75     True
8         327.75                    1 327.75          327.75     True
9         327.75                    1 327.75          327.75     True


In [6]:
# Check unique values in 'Order Item Quantity'
unique_order_item_qty = df['Order Item Quantity'].unique()
print(f"Unique values in 'Order Item Quantity': {unique_order_item_qty}")

Unique values in 'Order Item Quantity': [1 2 3 5 4]


In [7]:
expected_sales_per_customer = (
    (df_clean['product_price'] * df_clean['order_item_quantity'] )- df_clean['order_item_discount']
    
)

matches = np.isclose(df_clean['sales_per_customer'], expected_sales_per_customer, atol=0.01)

print(f"Matches: {matches.sum():,} / {len(matches):,}")
print(f"All rows match: {matches.all()}")

if not matches.all():
    print(df_clean.loc[~matches, [
        'product_price', 'order_item_discount', 'order_item_quantity', 'sales_per_customer'
    ]].head())

Matches: 180,519 / 180,519
All rows match: True


#### Proven relationship

- `sales = product_price * order_item_quantity`
- `sales_per_customer = (product_price * order_item_quantity) - order_item_discount`
- The discount amount and discount rate apply to the full line-item sales amount, not to a per-unit price
- Profit is also calculated for the total quantity, not for an individual product unit

##### rename columns for clarity


In [8]:
df_clean = df_clean.rename(columns={
    'sales': 'line_item_sales',
    'sales_per_customer': 'sales_after_discount',
    'order_item_profit_ratio':'profit_margin'
})

print(df_clean[['line_item_sales', 'sales_after_discount','profit_margin']].head())

   line_item_sales  sales_after_discount  profit_margin
0           327.75                314.64           0.29
1           327.75                311.36          -0.80
2           327.75                309.72          -0.80
3           327.75                304.81           0.08
4           327.75                298.25           0.45


### 3. Fix Data Types

Convert `shipping_date` to datetime. `order_date` is already datetime.

In [9]:
#recheck the dtypes
df_clean[['order_date','shipping_date']].dtypes

order_date       str
shipping_date    str
dtype: object

In [10]:
# Convert to datetime
df_clean['shipping_date'] = pd.to_datetime(df_clean['shipping_date'])

df_clean['order_date'] = pd.to_datetime(df_clean['order_date'])

# Verify
print(f"order_date dtype: {df_clean['order_date'].dtype}")
print(f"shipping_date dtype: {df_clean['shipping_date'].dtype}")
print(f"\nDate ranges:")
print(f"  Orders: {df_clean['order_date'].min()} → {df_clean['order_date'].max()}")
print(f"  Shipping: {df_clean['shipping_date'].min()} → {df_clean['shipping_date'].max()}")

order_date dtype: datetime64[us]
shipping_date dtype: datetime64[us]

Date ranges:
  Orders: 2015-01-01 00:00:00 → 2018-01-31 23:38:00
  Shipping: 2015-01-03 00:00:00 → 2018-02-06 22:14:00


### 4. Hande Null Values

In [11]:
# Check remaining nulls
nulls = df_clean.isnull().sum()
nulls = nulls[nulls > 0].sort_values(ascending=False)

if len(nulls) > 0:
    print("Columns with nulls:")
    for col, count in nulls.items():
        pct = (count / len(df_clean)) * 100
        print(f"  {col:25s} {count:>8,} ({pct:>5.2f}%)")
else:
    print("No nulls!")

Columns with nulls:
  order_zipcode              155,679 (86.24%)
  customer_zipcode                 3 ( 0.00%)


In [12]:
# ============================================================
# FEATURE ENGINEERING
# ============================================================

# 1. Shipping delay (negative = early, 0 = on time, positive = late)
df_clean['shipping_delay_days'] = (
    df_clean['days_shipping_real'] - df_clean['days_shipping_scheduled']
)

# 2. Discount impact %
df_clean['discount_pct'] = (
    (df_clean['line_item_sales'] - df_clean['sales_after_discount']) 
    / df_clean['line_item_sales'] * 100
).round(2)

# 3. Time-based features from order_date
df_clean['order_year'] = df_clean['order_date'].dt.year
df_clean['order_month'] = df_clean['order_date'].dt.month
df_clean['order_day_of_week'] = df_clean['order_date'].dt.day_name()
df_clean['order_year_month'] = df_clean['order_date'].dt.to_period('M').astype(str)

# 4. Profitability flag
df_clean['is_profitable'] = df_clean['profit_per_order'] > 0

df_clean['is_weekend'] = df_clean['order_day_of_week'].isin(['Saturday', 'Sunday'])


# Verify
print(f"Shape: {df_clean.shape}")
print(df_clean[['shipping_delay_days', 'discount_pct',
                'order_year_month', 'is_profitable']].head())

Shape: (180519, 46)
   shipping_delay_days  discount_pct order_year_month  is_profitable
0                   -1          4.00          2018-01           True
1                    1          5.00          2018-01          False
2                    0          5.50          2018-01          False
3                   -1          7.00          2018-01           True
4                   -2          9.00          2018-01           True


In [13]:
df_clean.describe().T

,count,mean,min,25%,50%,75%,max,std
days_shipping_real,"180,519.00",3.50,0.00,2.00,3.00,5.00,6.00,1.62
days_shipping_scheduled,"180,519.00",2.93,0.00,2.00,4.00,4.00,4.00,1.37
profit_per_order,"180,519.00",21.97,"-4,274.98",7.00,31.52,64.80,911.80,104.43
sales_after_discount,"180,519.00",183.11,7.49,104.38,163.99,247.40,"1,939.99",120.04
late_delivery_risk,"180,519.00",0.55,0.00,0.00,1.00,1.00,1.00,0.50
category_id,"180,519.00",31.85,2.00,18.00,29.00,45.00,76.00,15.64
customer_id,"180,519.00","6,691.38",1.00,"3,258.50","6,457.00","9,779.00","20,757.00","4,162.92"
customer_zipcode,"180,516.00","35,921.13",603.00,725.00,"19,380.00","78,207.00","99,205.00","37,542.46"
department_id,"180,519.00",5.44,2.00,4.00,5.00,7.00,12.00,1.63
order_date,180519,2016-06-12 17:47:04.669868,2015-01-01 00:00:00,2015-09-21 13:49:00,2016-06-11 13:06:00,2017-03-01 08:42:00,2018-01-31 23:38:00,NaN


In [14]:
late_delivery_unique = df_clean['late_delivery_risk'].unique()
print(f"Unique values in 'late_delivery_risk': {late_delivery_unique}")

unique_values_in_all_categorical_cols = df_clean.select_dtypes(include='object').columns
for col in unique_values_in_all_categorical_cols:
    unique_vals = df_clean[col].unique()
    print(f"Unique values in '{col}': {unique_vals}")

Unique values in 'late_delivery_risk': [0 1]
Unique values in 'payment_type': <StringArray>
['DEBIT', 'TRANSFER', 'CASH', 'PAYMENT']
Length: 4, dtype: str
Unique values in 'delivery_status': <StringArray>
['Advance shipping', 'Late delivery', 'Shipping on time', 'Shipping canceled']
Length: 4, dtype: str
Unique values in 'category_name': <StringArray>
[      'Sporting Goods',               'Cleats',        'Shop By Sport',      'Women's Apparel',          'Electronics',         'Boxing & MMA',     'Cardio Equipment',             'Trade-In',
     'Kids' Golf Clubs',   'Hunting & Shooting',  'Baseball & Softball',       'Men's Footwear',     'Camping & Hiking', 'Consumer Electronics',             'Cameras ',            'Computers',
           'Basketball',               'Soccer',       'Girls' Apparel',          'Accessories',     'Women's Clothing',               'Crafts',       'Men's Clothing',     'Tennis & Racquet',
  'Fitness Accessories',      'As Seen on  TV!',           'Golf Ba

C:\Users\nikhi\AppData\Local\Temp\ipykernel_5040\2640121198.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  unique_values_in_all_categorical_cols = df_clean.select_dtypes(include='object').columns


## 5. Feature Engineering

Create derived columns from existing data. These will be essential for analysis and dashboards.

| New Column | Derived From | Purpose |
|------------|--------------|---------|
| `is_late` | `late_delivery_risk` | Boolean for cleaner filtering |
| `shipping_delay_days` | `days_shipping_real - days_shipping_scheduled` | Quantify lateness |
| `profit_margin_pct` | `profit_per_order / sales` | % margin per order |
| `order_year`, `order_month`, `order_day_of_week` | `order_date` | Time-based analysis |
| `order_year_month` | `order_date` | Monthly trends |
| `is_profitable` | `profit_per_order > 0` | Quick loss filter |

In [15]:
# ============================================================
# FEATURE ENGINEERING
# ============================================================

# Shipping delay
df_clean['shipping_delay_days'] = (
    df_clean['days_shipping_real'] - df_clean['days_shipping_scheduled']
)

# Discount impact %
df_clean['discount_pct'] = (
    (df_clean['line_item_sales'] - df_clean['sales_after_discount']) 
    / df_clean['line_item_sales'] * 100
).round(2)

# Time-based
df_clean['order_year'] = df_clean['order_date'].dt.year
df_clean['order_month'] = df_clean['order_date'].dt.month
df_clean['order_day_of_week'] = df_clean['order_date'].dt.day_name()
df_clean['order_year_month'] = df_clean['order_date'].dt.to_period('M').astype(str)
df_clean['is_weekend'] = df_clean['order_day_of_week'].isin(['Saturday', 'Sunday'])

# Profitability
df_clean['is_profitable'] = df_clean['profit_per_order'] > 0



In [16]:
df_clean['order_value_tier'] = pd.qcut(df_clean['sales_after_discount'], q=[0, 0.33, 0.67, 1.0], labels=['Low', 'Medium', 'High'])

cutoffs = df_clean.groupby('order_value_tier', observed=True)['sales_after_discount'].agg(['min', 'max'])
print(cutoffs)

                    min      max
order_value_tier                
Low                7.49   118.29
Medium           118.30   205.00
High             205.03 1,939.99


In [17]:


# Delivery performance (cleaner labels)
performance_map = {
    'Advance shipping': 'Early',
    'Shipping on time': 'On Time',
    'Late delivery': 'Late',
    'Shipping canceled': 'Canceled'
}
df_clean['delivery_performance'] = df_clean['delivery_status'].map(performance_map)

print(f"Shape: {df_clean.shape}")
print(df_clean[['shipping_delay_days', 'discount_pct', 'is_weekend',
                'order_value_tier', 
                'delivery_performance', 'is_profitable']].head())

Shape: (180519, 48)
   shipping_delay_days  discount_pct  is_weekend order_value_tier delivery_performance  is_profitable
0                   -1          4.00       False             High                Early           True
1                    1          5.00        True             High                 Late          False
2                    0          5.50        True             High              On Time          False
3                   -1          7.00        True             High                Early           True
4                   -2          9.00        True             High                Early           True


In [19]:
# ============================================================
# SAVE CLEANED DATASET
# ============================================================

output_path = '../data/processed/dataco_cleaned.csv'
df_clean.to_csv(output_path, index=False)

print(f"✅ Saved: {output_path}")
print(f"Rows: {df_clean.shape[0]:,} | Columns: {df_clean.shape[1]}")
print(f"File size: ~{df_clean.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

✅ Saved: ../data/processed/dataco_cleaned.csv
Rows: 180,519 | Columns: 48
File size: ~230.6 MB


##  Cleaning Complete

### Transformations Applied
- Dropped **13 columns** (6 duplicates, 7 non-analytical/PII)
- Dropped **1 column** with heavy nulls (`order_zipcode`)
- Renamed all columns to `snake_case`
- Converted `shipping_date` to datetime
- Filled minor nulls in `customer_zipcode`
- Engineered **10 new features**:
  - `shipping_delay_days`, `discount_pct`, `is_profitable`
  - `order_year`, `order_month`, `order_day_of_week`, `order_year_month`, `is_weekend`
  - `order_value_tier`, `delivery_performance`

### Output
- **Final shape:** 180,519 rows × ~48 columns
- **Saved to:** `data/processed/dataco_cleaned.csv`

### Next Steps (Notebook 03 / SQL)
- Load cleaned CSV into PostgreSQL with normalized schema
- Begin business question analysis